# prep.tableS1

This table converts the McLeish et al. 2024 table S1 and S2 into a single Table that will be labelled as Table S1 in the article, aiming to provide:

- Library names
- Library site locations
- Library number of extracts
- Library host taxon

Among others. 


In [1]:
import pandas as pd
import seaborn as sns 
from yaml import load, Loader
from daforfer import DaforferDB
conf = load(open("conf.yaml"), Loader)
db = DaforferDB(conf['database'])
si = DaforferDB(conf['si'])
db.toc()

┌──────────────────────┬────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────┐
│         name         │                                                                                  description                                                                                   │
│       varchar        │                                                                                    varchar                                                                                     │
├──────────────────────┼────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────┤
│ D_sites              │ This table contains key information about each of the libraries, such as their site, habitat and host                                                                  

## Load libraries and sampling tables (S1 and S2)

In [2]:
libraries = pd.read_csv("input/mcleish24.TableS2.csv", sep=';')
sampling = pd.read_csv("input/mcleish24.TableS1.csv", sep=';')


In [3]:
libraries

,Library_code,Host_taxon,Habitat,Collection_code,No_extracts
0,PV001,Amaranthus sp,Crop,M1V,8
1,PV002,Convolvulus arvensis,Crop,M1V,11
2,PV003,Cucumis melo,Crop,M1V,13
3,PV003bgi,Cucumis melo,Crop,M1V,13
4,PV004bgi,Cyperus longus,Crop,M1V,7
...,...,...,...,...,...
318,PV586,Fumaria parviflora,Crop,H1P,8
319,PV587,Hirschfeldia incana,Crop,H1P,8
320,PV588,Hordeum vulgare,Crop,H1P,8
321,PV589,Hordeum vulgare,Crop,H1P,8


In [4]:
sampling

,Date,Site_code,Collection_code,Times_sampled,Season,Location,Longitude,Latitude
0,1/2/16,C1,C1F,1,autumn,Aranjuez,-3.593308,40.051302
1,30/1/17,C1,C3F,2,autumn,Aranjuez,-3.593308,40.051302
2,1/2/16,C2,C2F,1,autumn,Aranjuez,-3.599064,40.043193
3,30/1/17,C2,C4F,2,autumn,Aranjuez,-3.599064,40.043193
4,19/11/15,E1,E1F,1,autumn,Aranjuez,-3.500323,40.059138
...,...,...,...,...,...,...,...,...
73,3/5/17,Q4,Q4P,2,spring,Mondéjar,-3.137145,40.345348
74,16/7/15,Z1,Z1V,1,summer,Villaconejos,-3.476076,40.050052
75,20/7/16,Z1,Z3V,1,summer,Villaconejos,-3.476076,40.050052
76,24/7/15,Z2,Z2V,1,summer,Villamanrique de Tajo,-3.131000,40.044720


## Merge

The merge process requires dealing with libraries that were pooled from various sites. 

In [5]:
libraries['Collection_code'] = libraries['Collection_code'].apply(lambda x: x.split("_"))
tmp = libraries.drop_duplicates(subset=['Library_code'])[['Library_code', 'Collection_code']].to_dict(orient='records')
library_collection_code_list = []
for item in tmp:
    for collection_code in item['Collection_code']:
        library_collection_code_list.append({"Library_code": item['Library_code'], "Collection_code": collection_code})
library_collection_code = pd.DataFrame.from_records(library_collection_code_list)
library_collection_code

,Library_code,Collection_code
0,PV001,M1V
1,PV002,M1V
2,PV003,M1V
3,PV003bgi,M1V
4,PV004bgi,M1V
...,...,...
330,PV586,H1P
331,PV587,H1P
332,PV588,H1P
333,PV589,H1P


In [6]:
sampling['Date'] = pd.to_datetime(sampling['Date'])
sampling_deduplicated = sampling.sort_values(by='Date').drop_duplicates('Collection_code')

/var/folders/xl/z7y434d524s8xpvqnb0gst580000gn/T/ipykernel_48498/3501850429.py:1: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  sampling['Date'] = pd.to_datetime(sampling['Date'])


In [7]:
library_site = pd.merge(
    library_collection_code, 
    sampling_deduplicated, 
    on='Collection_code'
)

library_site


,Library_code,Collection_code,Date,Site_code,Times_sampled,Season,Location,Longitude,Latitude
0,PV001,M1V,2015-07-16,M1,1,summer,Aranjuez,-3.345220,40.031840
1,PV002,M1V,2015-07-16,M1,1,summer,Aranjuez,-3.345220,40.031840
2,PV003,M1V,2015-07-16,M1,1,summer,Aranjuez,-3.345220,40.031840
3,PV003bgi,M1V,2015-07-16,M1,1,summer,Aranjuez,-3.345220,40.031840
4,PV004bgi,M1V,2015-07-16,M1,1,summer,Aranjuez,-3.345220,40.031840
...,...,...,...,...,...,...,...,...,...
330,PV586,H1P,2016-04-21,H1,1,spring,Villaconejos,-3.477574,40.049330
331,PV587,H1P,2016-04-21,H1,1,spring,Villaconejos,-3.477574,40.049330
332,PV588,H1P,2016-04-21,H1,1,spring,Villaconejos,-3.477574,40.049330
333,PV589,H1P,2016-04-21,H1,1,spring,Villaconejos,-3.477574,40.049330


In [8]:
library_site.value_counts('Library_code')

Library_code
PV578    3
PV570    3
PV582    2
PV566    2
PV562    2
        ..
PV115    1
PV114    1
PV113    1
PV112    1
PV590    1
Name: count, Length: 323, dtype: int64

In [9]:
tableSiteInformation = pd.merge(
    library_site,
    libraries[['Library_code', 'Habitat', 'No_extracts', 'Host_taxon']], on='Library_code', how='left'
)
# sites['Habitat_type'] = sites['Habitat'].map(sites_to_type)
# sites = sites.drop_duplicates(subset=['Site_code'])[['Site_code', 'Longitude', 'Latitude', 'Location', 'Habitat', 'Habitat_type']].to_dict(orient='records')
tableSiteInformation = tableSiteInformation[['Library_code',  'Host_taxon', 'No_extracts', 'Site_code', 'Habitat', 'Longitude', 'Latitude', 'Location', 'Date']]
tableSiteInformation = tableSiteInformation.rename(columns={'Site_code': 'site', 'Library_code': 'library', 'Habitat': 'habitat', 'No_extracts': 'n_extracts', 'Host_taxon':'host_taxon', 'Longitude': 'longitude', 'Latitude':'latitude', 'Location':'location', 'Date':'date'})
tableSiteInformation = tableSiteInformation.sort_values(by=['site', 'library'])
tableSiteInformation['l'] = tableSiteInformation['library']
tableSiteInformation = tableSiteInformation.set_index('library')
tableSiteInformation['date'] = tableSiteInformation['date'].dt.strftime('%Y-%m-%d')
tableSiteInformation = tableSiteInformation.groupby(['l'], as_index=True).transform(lambda x : ";".join([str(m) for m in x.unique()])).reset_index().drop_duplicates(subset=['library'])
tableSiteInformation

,library,host_taxon,n_extracts,site,habitat,longitude,latitude,location,date
0,PV534,Diplotaxis erucoides,3,C1,Crop,-3.593308,40.051302,Aranjuez,2016-01-02
1,PV535,Brassica oleracea,17,C1,Crop,-3.593308,40.051302,Aranjuez,2016-01-02
2,PV538,Brassica oleracea,8,C1,Crop,-3.593308,40.051302,Aranjuez,2016-01-02
3,PV540,Picris echioides,1,C1,Crop,-3.593308,40.051302,Aranjuez,2016-01-02
4,PV544,Sisymbrium runcinatum,4,C1,Crop,-3.593308,40.051302,Aranjuez,2016-01-02
...,...,...,...,...,...,...,...,...,...
330,PV590,Zea mays,11,Z1,Crop,-3.476076,40.050052,Villaconejos,2015-07-16
331,PV047,Zea mays,13,Z2,Crop,-3.131,40.04472,Villamanrique de Tajo,2015-07-24
332,PV048,Desconocida 4,9,Z2,Crop,-3.131,40.04472,Villamanrique de Tajo,2015-07-24
333,PV527,Convolvulus arvensis,4,Z2,Crop,-3.131,40.04472,Villamanrique de Tajo,2015-07-24


In [10]:
db.save_dataframe(
    tableSiteInformation, table_name="d_TableS1", 
    description="Table S1: Library sites and context"
)

Saved d_TableS1 to db.2026-02-24


In [11]:
si.save_dataframe(
    tableSiteInformation, table_name="tableSiteInformation", 
    description="Library sites and context"
)

Saved tableSiteInformation to si.2026-02-24


In [12]:
si.conn.close()
db.conn.close()